# MO433 on Colab

Thin runner: all the code lives in the repository and every step is a `make` target.
Run the cells from top to bottom; only the **last cell** changes from session to session.

Before the first run:
1. *Runtime → Change runtime type → GPU: **L4*** (every timed run must use the same GPU, DECISOES 30).
2. *Secrets* (key icon on the left) → add `GITHUB_TOKEN` (fine-grained, read-only access to
   `mo433-2s2026`) and enable *Notebook access*.

On Colab, `make` only builds what is **missing** on Drive: after cloning, the code files get
an old fixed date, so they never look newer than the data. To redo a step, delete its output
file. Never use `make -B` here: it rebuilds the whole chain of prerequisites.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["MO433_DATA"] = "/content/drive/MyDrive/mo433-2s2026"
assert os.path.isdir(os.environ["MO433_DATA"]), "Drive folder mo433-2s2026 not found"

In [ ]:
# Clone (or update) the private repository with the token from Colab Secrets.
# The token is never printed and is not left in the remote URL.
import subprocess
from google.colab import userdata

REPO = "github.com/helen-fornazier/mo433-2s2026.git"
CODE = "/content/mo433-2s2026"
token = userdata.get("GITHUB_TOKEN")
authed = f"https://x-access-token:{token}@{REPO}"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.replace(token, "***"))
    return r.stdout.strip()

if not os.path.isdir(CODE):
    git("clone", "-q", authed, CODE)
    git("remote", "set-url", "origin", f"https://{REPO}", cwd=CODE)
else:
    git("pull", "-q", authed, "main", cwd=CODE)
print("commit:", git("log", "--oneline", "-1", cwd=CODE))

# Old fixed date on every code file: make then only builds outputs that are missing on Drive.
!find {CODE} -path {CODE}/.git -prune -o -exec touch -h -d 2000-01-01 {{}} +
%cd {CODE}

In [ ]:
# Colab's own torch/torchvision (CUDA build) are kept; the pinned libraries are installed.
!pip install -q -r requirements.txt
!pip install -q -e .
!make env-info

## Run a step

Set `TARGET` to the step to run (see `make help` and the plan). `make -n` first shows what
would run; check it before spending GPU time.

In [ ]:
TARGET = "env-info"
!make -n {TARGET}

In [ ]:
!make {TARGET}